In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Sales Analytics") \
    .getOrCreate()

In [3]:
from google.colab import files

uploaded = files.upload()

Saving store_sales_transactions.csv to store_sales_transactions.csv
Saving retail_products_master.csv to retail_products_master.csv


In [4]:
products = spark.read.csv(
    "retail_products_master.csv",
    header=True,
    inferSchema=True
)

sales = spark.read.csv(
    "store_sales_transactions.csv",
    header=True,
    inferSchema=True
)

In [5]:
products.show()

sales.show()

+----------+------------+-----------+----------+-------------+
|product_id|product_name|   category|cost_price|selling_price|
+----------+------------+-----------+----------+-------------+
|      P101|      Laptop|Electronics|     60000|        70000|
|      P102|       Mouse|Electronics|       300|          500|
|      P103|    Keyboard|Electronics|       900|         1500|
|      P104|     Monitor|Electronics|      9000|        12000|
|      P105|       Chair|  Furniture|      2500|         4000|
|      P106|       Table|  Furniture|      4500|         7000|
|      P107|    Notebook| Stationery|        40|           70|
|      P108|         Pen| Stationery|         5|           10|
+----------+------------+-----------+----------+-------------+

+-------+----------+---------+--------+
|sale_id|product_id|    store|quantity|
+-------+----------+---------+--------+
|   S001|      P101|  Chennai|       2|
|   S002|      P102|  Chennai|       8|
|   S003|      P103|Bangalore|       5|
|  

In [6]:
products.printSchema()

sales.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- cost_price: integer (nullable = true)
 |-- selling_price: integer (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store: string (nullable = true)
 |-- quantity: integer (nullable = true)



In [7]:
products = products.dropna()

sales = sales.dropna()

In [8]:
final_df = sales.join(
    products,
    on="product_id",
    how="inner"
)

final_df.show()

+----------+-------+---------+--------+------------+-----------+----------+-------------+
|product_id|sale_id|    store|quantity|product_name|   category|cost_price|selling_price|
+----------+-------+---------+--------+------------+-----------+----------+-------------+
|      P101|   S009|Bangalore|       1|      Laptop|Electronics|     60000|        70000|
|      P101|   S001|  Chennai|       2|      Laptop|Electronics|     60000|        70000|
|      P102|   S011|Bangalore|       5|       Mouse|Electronics|       300|          500|
|      P102|   S002|  Chennai|       8|       Mouse|Electronics|       300|          500|
|      P103|   S003|Bangalore|       5|    Keyboard|Electronics|       900|         1500|
|      P104|   S010|Hyderabad|       2|     Monitor|Electronics|      9000|        12000|
|      P104|   S004|Hyderabad|       1|     Monitor|Electronics|      9000|        12000|
|      P105|   S012|Hyderabad|       1|       Chair|  Furniture|      2500|         4000|
|      P10

In [9]:
from pyspark.sql.functions import *

final_df = final_df.withColumn(
    "Revenue",
    col("selling_price") * col("quantity")
)

final_df = final_df.withColumn(
    "Profit",
    (col("selling_price") - col("cost_price")) * col("quantity")
)

final_df.show()

+----------+-------+---------+--------+------------+-----------+----------+-------------+-------+------+
|product_id|sale_id|    store|quantity|product_name|   category|cost_price|selling_price|Revenue|Profit|
+----------+-------+---------+--------+------------+-----------+----------+-------------+-------+------+
|      P101|   S009|Bangalore|       1|      Laptop|Electronics|     60000|        70000|  70000| 10000|
|      P101|   S001|  Chennai|       2|      Laptop|Electronics|     60000|        70000| 140000| 20000|
|      P102|   S011|Bangalore|       5|       Mouse|Electronics|       300|          500|   2500|  1000|
|      P102|   S002|  Chennai|       8|       Mouse|Electronics|       300|          500|   4000|  1600|
|      P103|   S003|Bangalore|       5|    Keyboard|Electronics|       900|         1500|   7500|  3000|
|      P104|   S010|Hyderabad|       2|     Monitor|Electronics|      9000|        12000|  24000|  6000|
|      P104|   S004|Hyderabad|       1|     Monitor|Ele

In [15]:
profit_margin_df = final_df.groupBy("category").agg(
    sum("Revenue").alias("Total_Revenue"),
    sum("Profit").alias("Total_Profit")
)

profit_margin_df = profit_margin_df.withColumn(
    "Profit_Margin_Percent",
    round((col("Total_Profit") / col("Total_Revenue")) * 100, 2)
)

profit_margin_df.show()

+-----------+-------------+------------+---------------------+
|   category|Total_Revenue|Total_Profit|Profit_Margin_Percent|
+-----------+-------------+------------+---------------------+
| Stationery|         3800|        1700|                44.74|
|Electronics|       260000|       44600|                17.15|
|  Furniture|        30000|       11000|                36.67|
+-----------+-------------+------------+---------------------+



In [16]:
profit_margin_df.toPandas().to_csv(
    "profit_margin_by_category.csv",
    index=False
)

In [17]:
final_df.createOrReplaceTempView("sales_data")

In [18]:
spark.sql("""
SELECT
    product_name,
    SUM(quantity) AS total_quantity_sold
FROM sales_data
GROUP BY product_name
ORDER BY total_quantity_sold DESC
LIMIT 3
""").show()

+------------+-------------------+
|product_name|total_quantity_sold|
+------------+-------------------+
|         Pen|                100|
|    Notebook|                 40|
|       Mouse|                 13|
+------------+-------------------+



In [19]:
dashboard_df = final_df.groupBy("store").agg(
    sum("Revenue").alias("Total_Revenue"),
    sum("Profit").alias("Total_Profit"),
    avg("Profit").alias("Average_Profit")
)

dashboard_df.show()

+---------+-------------+------------+--------------+
|    store|Total_Revenue|Total_Profit|Average_Profit|
+---------+-------------+------------+--------------+
|Bangalore|        94000|       19000|        4750.0|
|  Chennai|       157000|       26600|        6650.0|
|Hyderabad|        42800|       11700|        2925.0|
+---------+-------------+------------+--------------+



In [20]:
dashboard_df.toPandas().to_csv(
    "dashboard_store_metrics.csv",
    index=False
)

In [21]:
from google.colab import files

files.download("profit_margin_by_category.csv")
files.download("dashboard_store_metrics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>